In [1]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = 8096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = True, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/duckq1u/miniconda3/envs/ocr2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.6: Fast Llama patching. Transformers: 4.55.2.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.469 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### Fine-tunning Optionals 

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r=16,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)

## Form maker

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2",
)

#### Load dataset

In [ ]:

import os
import json
from datasets import Dataset
from prompt_cached import get_prompt
# Loop on folder for read folders


def read_txt(file_path: str) -> str:
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    return content


def read_json(file_path: str, is_schema=False) -> str:
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    
    if is_schema:
        content = data
    else:
        content = data[0]['output']
    
    # Remove unwanted keys from the dictionary before converting to string
    remove_keys = ["TaiSan", "NoiDungThayDoi", "MoTaChungTaiSan"]
    for key in remove_keys:
        content.pop(key, None)
    
    # Convert to JSON string
    return json.dumps(content, ensure_ascii=False, indent=2)

# follow this format schema below:\n <schema>{str(read_json(schema_path))}\n</schema> </requirement>
def load_dataset() -> list:
    dataset = []
    path = "/home/duckq1u/Downloads/FIL/dataset/Training"

    for folder_name in os.listdir(path):
        for file_name in os.listdir(os.path.join(path, folder_name)):
            file_path = os.path.join(path, folder_name, file_name.split(".")[0])

            conversation = {
                "conversations": [
                    {
                        "content": f"{get_prompt()}",
                        "role": "system",
                    },
                    {
                        "content": f"\n**TÀI LIỆU MARKDOWN**:\n{str(read_txt(file_path=file_path+'.txt'))}\n",
                        "role": "user",
                    },
                    {
                        "content": f"{str(read_json(file_path=file_path+'.json'))}\n",
                        "role": "assistant",
                    },
                ]
            }
            # print(type(read_json(file_path=file_path+'.json')))
            # break
            dataset.append(conversation)
    return dataset

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {
        "text": texts,
    }

ds = Dataset.from_list(load_dataset()).train_test_split(test_size=0.1)
dataset=ds['train'].map(formatting_prompts_func, batched=True) 
test_dataset=ds['test'].map(formatting_prompts_func, batched=True) 

## Config trainer

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = test_dataset, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 4,
        # max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
        eval_strategy="steps",
        save_steps=10,
        eval_steps=2,
        save_total_limit=20, # Save only the last checkpoint
    ),
)

 Unsloth's train_on_completions method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

test decoder

In [ ]:
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"]))

Result from batch

In [ ]:
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " "))

In [ ]:
# trainer_stats = trainer.train()